In [1]:
from Bio import SeqIO
import pandas as pd

In [ ]:
gff_ref = '/home/marymardegan/documents_marymardegan/benchannot_pos/results/drosophila_melanogaster/reference/genomic.gff'
faa_ref = '/home/marymardegan/documents_marymardegan/benchannot_pos/results/drosophila_melanogaster/reference/protein.faa'
output_faa_path = "/home/marymardegan/documents_marymardegan/benchannot_pos/results/drosophila_melanogaster/reference/drosophila_reference.faa"


In [3]:
# create mapping from NP_* to rna-NM_*
np_for_rna = {}  # maping from NP_ IDs to rna-NM_ IDs
with open (gff_ref, "r") as gff:
    for line in gff:
        if line.startswith("#") or "\tCDS\t" not in line: # Ignore comments and non-CDS lines
            continue
        fields = line.strip().split("\t")  # Split the GFF line into fields
        attr_field = fields[8]  # Attribute field is the 9th column (index 8)
        attrs = dict(item.split("=") for item in attr_field.split(";") if "=" in item)  # Split attributes into key-value pairs
        if "ID" in attrs and "Parent" in attrs:
            np_id = attrs["ID"].replace("cds-", "").split(";")[0]  # Extract the CDS ID (ex.: NP_009332.1)
            rna_id = attrs["Parent"].split(":")[0]               # Extract the RNA ID (ex.: rna-NM_001180043.1)
            np_for_rna[np_id] = rna_id                              # Map NP_* -> rna-NM_*

# Step 2: Write new FASTA file with updated IDs
updated_records = []
for record in SeqIO.parse(faa_ref, "fasta"):
    orig_id = record.id
    new_id = np_for_rna.get(orig_id)
    if new_id:
        if record.description.startswith(orig_id):
            record.description = record.description.replace(orig_id, new_id, 1)
        record.id = new_id
        updated_records.append(record)

# Step 3: Save the updated records to a new FASTA file
with open(output_faa_path, "w") as out_fasta:
    SeqIO.write(updated_records, out_fasta, "fasta")

print(f"Arquivo reescrito com {len(updated_records)} sequências usando IDs rna")

Arquivo reescrito com 30802 sequências usando IDs rna


In [4]:
# Step 4: Create a TSV file with RNA_ID and Description
reff_fasta = output_faa_path
records = []

for record in SeqIO.parse(reff_fasta, "fasta"): # Reads the FASTA file
    header = record.description # Gets the complete header
    parts = header.split(" ", 1) # Split the header into parts
    rna_id = parts[0].replace(">", "").strip() # Keep only the ID
    desc = parts[1] if len(parts) > 1 else ""
    desc = desc.replace("[Drosophila melanogaster]", "").strip() # Remove the species from the description
    records.append({"RNA_ID": rna_id, "Description": desc})

reff_df = pd.DataFrame(records)

# Output path
reff_tsv = "drosophila_melanogaster.tsv"

# Save with commented header
with open(reff_tsv, "w", encoding="utf-8", newline="") as f:
    f.write("# RNA_ID\tDescription\n")  # Writes the header as a comment
    reff_df.to_csv(f, sep="\t", index=False, header=False)

In [5]:
# Here we separate the isoforms into a new column, keeping the original description clean
reff_isoforms = reff_df.copy()
reff_isoforms["Isoforms"] = reff_isoforms["Description"].str.split(", isoform", n=1).str[1] # Extract isoforms if they exist and remove them from the description
reff_isoforms["Description"] = reff_isoforms["Description"].str.split(", isoform", n=1).str[0].str.strip() # Keep only the main description without isoforms
reff_isoforms.to_csv("drosophila_melanogaster_separated.tsv", sep="\t", index=False) # Save the new TSV with separated isoforms
reff_isoforms

,RNA_ID,Description,Isoforms
0,rna-NM_001007095.3,uncharacterized protein Dmel_CG42637,C
1,rna-NM_001007096.3,uncharacterized protein Dmel_CG42637,B
2,rna-NM_001014453.2,phosphatase 1 nuclear targeting subunit,D
3,rna-NM_001014454.2,phosphatase 1 nuclear targeting subunit,A
4,rna-NM_001014455.2,phosphatase 1 nuclear targeting subunit,B
...,...,...,...
30797,rna-Dmel_CG34085,NADH dehydrogenase subunit 4 (mitochondrion),NaN
30798,rna-Dmel_CG34086,NADH dehydrogenase subunit 4L (mitochondrion),NaN
30799,rna-Dmel_CG34089,NADH dehydrogenase subunit 6 (mitochondrion),NaN
30800,rna-Dmel_CG34090,cytochrome b (mitochondrion),NaN


In [6]:
reff_wi = reff_isoforms.drop(columns=["Isoforms"])
reff_wi.to_csv("drosophila_melanogaster_without_isoforms.tsv", sep="\t", index=False) # Save the new TSV without isoforms

In [7]:
df = reff_isoforms.copy()

# Marca descrições repetidas
desc_repetida = df.duplicated(subset=['Description'], keep=False)

# Marca grupos em que existe pelo menos uma isoforma preenchida
tem_isoforma_no_grupo = df.groupby('Description')['Isoforms'].transform(lambda x: x.notna().any())

# Só agrupa quando as duas condições são verdadeiras
mask_agrupar = desc_repetida & tem_isoforma_no_grupo

# Parte que vai ficar como está
df_nao_agrupar = df[~mask_agrupar].copy()

# Parte que vai ser agrupada
df_agrupar = (
    df[mask_agrupar]
    .groupby('Description', as_index=False)
    .agg({
        'RNA_ID': lambda x: ', '.join(pd.unique(x.astype(str))),
        'Isoforms': lambda x: ', '.join(pd.unique(x.dropna().astype(str)))
    })
)

# Junta tudo de volta
df_final = pd.concat([df_nao_agrupar, df_agrupar], ignore_index=True)

df_final.to_csv("drosophila_melanogaster_grouped.tsv", sep="\t", index=False)
df_final

df_final_no_isoforms = df_final.drop(columns=['Isoforms'])
df_final_no_isoforms.to_csv("drosophila_melanogaster_grouped_no_isoforms.tsv", sep="\t", index=False)
df_final_no_isoforms

,RNA_ID,Description
0,rna-NM_001014456.1,gustatory receptor 22b
1,rna-NM_001014458.5,uncharacterized protein Dmel_CG33543
2,rna-NM_001014488.2,uncharacterized protein Dmel_CG33552
3,rna-NM_001014492.2,uncharacterized protein Dmel_CG33511
4,rna-NM_001014493.3,uncharacterized protein Dmel_CG33510
...,...,...
13994,"rna-NM_001299650.1, rna-NM_001299651.1, rna-NM...",zonda
13995,"rna-NM_001043112.2, rna-NM_001043113.2, rna-NM...",zormin
13996,"rna-NM_001298527.1, rna-NM_078687.2, rna-NM_16...",zwischenferment
13997,"rna-NM_001015260.3, rna-NM_001015261.3, rna-NM...",zydeco


In [8]:
grouped_id = reff_isoforms.copy()
grouped_id = (
    grouped_id.drop(columns=["Isoforms"])
    .groupby("Description", as_index=False)
    .agg({"RNA_ID": lambda x: ", ".join(map(str, x))})
)
grouped_id.to_csv("drosophila_melanogaster_grouped.tsv", sep="\t", index=False)
grouped_id

,Description,RNA_ID
0,(6-4)-photolyase,"rna-NM_001273703.1, rna-NM_001273704.1, rna-NM..."
1,(A+T)-stretch binding protein,rna-NM_001111011.3
2,"1,4-Alpha-Glucan branching enzyme",rna-NM_176162.3
3,1-Acylglycerol-3-phosphate O-acyltransferase 1,"rna-NM_001272565.1, rna-NM_132600.2, rna-NM_16..."
4,1-Acylglycerol-3-phosphate O-acyltransferase 2,"rna-NM_001169441.2, rna-NM_001298826.1, rna-NM..."
...,...,...
13875,zwilch,rna-NM_170542.2
13876,zwischenferment,"rna-NM_001298527.1, rna-NM_078687.2, rna-NM_16..."
13877,zydeco,"rna-NM_001015260.3, rna-NM_001015261.3, rna-NM..."
13878,zye,rna-NM_140963.2


In [9]:
# For better analysis we will generate a version of the grouped file with only one RNA_ID per description, keeping the first one listed in the original file

grouped_id_unique = grouped_id.copy()
grouped_id_unique["RNA_ID"] = grouped_id_unique["RNA_ID"].str.split(", ", n=1).str[0].str.strip()
grouped_id_unique.to_csv("drosophila_melanogaster_grouped_unique.tsv", sep="\t", index=False)
grouped_id_unique

,Description,RNA_ID
0,(6-4)-photolyase,rna-NM_001273703.1
1,(A+T)-stretch binding protein,rna-NM_001111011.3
2,"1,4-Alpha-Glucan branching enzyme",rna-NM_176162.3
3,1-Acylglycerol-3-phosphate O-acyltransferase 1,rna-NM_001272565.1
4,1-Acylglycerol-3-phosphate O-acyltransferase 2,rna-NM_001169441.2
...,...,...
13875,zwilch,rna-NM_170542.2
13876,zwischenferment,rna-NM_001298527.1
13877,zydeco,rna-NM_001015260.3
13878,zye,rna-NM_140963.2
